# 00 — Data Preparation & EDA

This notebook:
1. Clones the repo and sets up paths (Colab-compatible)
2. Loads the pre-processed JSONL splits
3. Provides exploratory analysis of the JTBD classification dataset

**Run locally first**: `python scripts/prepare_dataset.py` → commit → push → then this notebook just loads from `data/processed/`.

In [1]:
# ── 0. Environment setup (idempotent — safe to re-run) ────────────────────────
import sys, os

try:
    import google.colab
    IN_COLAB = True
    print('Running on Google Colab')
except ImportError:
    IN_COLAB = False
    print('Running locally')

REPO_URL = 'https://github.com/mikhailarutyunov/text-to-lora-demo.git'  # update if needed
REPO_NAME = 'text-to-lora-demo'

if IN_COLAB:
    # Clone or pull repo
    if not os.path.exists(f'/content/{REPO_NAME}'):
        !git clone {REPO_URL} /content/{REPO_NAME}
    else:
        !cd /content/{REPO_NAME} && git pull
    %cd /content/{REPO_NAME}

    # Install from requirements file (torch pre-installed on Colab — excluded)
    !pip install -q -r requirements_colab.txt
else:
    # Local: move to repo root (assumes notebook is in notebooks/)
    repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    os.chdir(repo_root)

# Add src/ to path
if 'src' not in sys.path:
    sys.path.insert(0, 'src')

print(f'Working directory: {os.getcwd()}')

Running on Google Colab
Cloning into '/content/text-to-lora-demo'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: '/content/text-to-lora-demo'
/content
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements_colab.txt'
Working directory: /content


In [ ]:
# ── 1. Load splits ─────────────────────────────────────────────────────────────
from data_utils import load_splits, NODE_TYPES, LABEL2ID

splits = load_splits('data/processed')
train, val, test = splits['train'], splits['val'], splits['test']

print(f'Train: {len(train):4d} examples')
print(f'Val:   {len(val):4d} examples')
print(f'Test:  {len(test):4d} examples')
print(f'Total: {len(train)+len(val)+len(test):4d} examples')

In [ ]:
# ── 2. Class distribution ──────────────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

all_examples = train + val + test
type_counts = Counter(e['node_type'] for e in all_examples)

df_dist = pd.DataFrame([
    {
        'node_type': nt,
        'count': type_counts.get(nt, 0),
        'pct': type_counts.get(nt, 0) / len(all_examples) * 100
    }
    for nt in NODE_TYPES
]).sort_values('count', ascending=False)

print(df_dist.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=df_dist, x='node_type', y='count', ax=ax, palette='Blues_d')
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right')
ax.set_title('Node type distribution — full dataset')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

# Imbalance ratio
max_c = df_dist['count'].max()
min_c = df_dist[df_dist['count'] > 0]['count'].min()
print(f'\nImbalance ratio (max/min): {max_c/min_c:.1f}x')

In [ ]:
# ── 3. Per-split class distribution heatmap ───────────────────────────────────
import numpy as np

split_dists = {}
for split_name, exs in splits.items():
    counts = Counter(e['node_type'] for e in exs)
    split_dists[split_name] = [counts.get(nt, 0) for nt in NODE_TYPES]

df_heat = pd.DataFrame(split_dists, index=NODE_TYPES)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(df_heat, annot=True, fmt='d', cmap='YlOrRd', ax=ax)
ax.set_title('Examples per class per split')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4. Quote and context length distributions ─────────────────────────────────
df_all = pd.DataFrame(all_examples)
df_all['quote_len'] = df_all['quote'].str.len()
df_all['context_len'] = df_all['context'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df_all['quote_len'], bins=40, ax=axes[0], color='steelblue')
axes[0].set_title('Quote length (chars)')
axes[0].axvline(df_all['quote_len'].median(), color='red', linestyle='--',
                label=f'median={df_all["quote_len"].median():.0f}')
axes[0].legend()

sns.histplot(df_all['context_len'], bins=40, ax=axes[1], color='teal')
axes[1].set_title('Context utterance length (chars)')
axes[1].axvline(df_all['context_len'].median(), color='red', linestyle='--',
                label=f'median={df_all["context_len"].median():.0f}')
axes[1].legend()

plt.tight_layout()
plt.show()

print('\nSummary statistics:')
print(df_all[['quote_len', 'context_len']].describe().round(1))

In [ ]:
# ── 5. Spot-check examples ────────────────────────────────────────────────────
from data_utils import format_prompt
import random

random.seed(0)
sample = random.sample(train, min(5, len(train)))

for i, ex in enumerate(sample, 1):
    print(f'--- Example {i} [{ex["node_type"]}] ---')
    print(format_prompt(ex))
    print()

In [ ]:
# ── 6. Missing context audit ──────────────────────────────────────────────────
no_ctx = [e for e in all_examples if not e.get('context', '').strip()]
pct_no_ctx = len(no_ctx) / len(all_examples) * 100
print(f'Examples without context: {len(no_ctx)} ({pct_no_ctx:.1f}%)')
if no_ctx:
    print('\nBreakdown by node_type:')
    for nt, c in Counter(e['node_type'] for e in no_ctx).most_common():
        print(f'  {nt}: {c}')